In [ ]:
import os
import pandas as pd
import shutil

from tqdm import tqdm
import random
import json

from metamoth import parse_metadata
import soundfile as sf

## 0. Utilities

In [ ]:
# Check if a file is between two time bounds
def isFilenameInBounds(filename, bounds):
    start, finish = bounds.split()

    time_start_date, time_start_time = start.split("_")
    time_start = int(time_start_date) * 1000000 + int(time_start_time)

    time_finish_date, time_finish_time = finish.split("_")
    time_finish = int(time_finish_date) * 1000000 + int(time_finish_time)

    filename_date, filename_time = filename.split(".")[0].split("_")
    time_filename = int(filename_date) * 1000000 + int(filename_time)

    return (time_filename >= time_start and time_filename <= time_finish)

# Check if a file is within one of a list of possible time bounds pairs
def isFilenameInPeriods(filename, time_periods):
    for bounds in time_periods:
        if isFilenameInBounds(filename, bounds):
            return True, time_periods[bounds]
    return False, None

# Check if file corresponds to the 1st or 31st minute of an hour
def isHalfHour(filename):
    filename_date, filename_time = filename.split(".")[0].split("_")
    return filename_time[2:4] in ['00','30']

In [ ]:
# Dictionary matching time bounds to raw audio folder
TIME_PERIODS = {
    '20251113_150000 20251208_000000':r'',
    '20251209_153500 20251231_125300':r'',
    '20251231_145600 20260116_123800':r'',
    '20260116_144500 20260209_123600':r'',
    '20260209_144200 20260304_122800':r''
}

BATCH_SIZE = 50
SEED = 100
SPLIT_FOLDER = r''
HYDROPHONE_ID = 'NessB'

## 1. Make a list of valid, half-hour files

In [ ]:
filelist = []

for input_dir in TIME_PERIODS.values():

    dir_list = [f for f in os.listdir(input_dir) if (f.endswith('.WAV') and f.startswith('20'))]

    for filename in tqdm(dir_list, total = len(dir_list)):
        in_bounds, _ = isFilenameInPeriods(filename, TIME_PERIODS)
        if in_bounds and isHalfHour(filename):
            filelist.append(filename)

database = []

sorted_filelist = sorted(filelist)

## 2. Copy these files in an adequate structure

In [ ]:
random.seed(SEED)
random.shuffle(sorted_filelist)

# Divide all files into batches of length BATCH_SIZE
batch_dictionary = {i : [] for i in  range(len(sorted_filelist)//BATCH_SIZE + 1)}

for i, filename in enumerate(sorted_filelist):
    _, dir = isFilenameInPeriods(filename, TIME_PERIODS)
    batch_dictionary[i//BATCH_SIZE].append([dir, filename])


for batch_number, batch_list in tqdm(batch_dictionary.items()):

    path = os.path.join(SPLIT_FOLDER,"Split_"+str(batch_number))
    os.mkdir(path)

    for in_batch_index, coordinates in enumerate(batch_list):
        dir, filename = coordinates
        
        # Add index at beginning of file name so that, when opened in Raven Pro, files are ordered following
        # a pre-set random order rather than temporal order
        random_indexed_filename = str(batch_number * BATCH_SIZE + in_batch_index)+'_'+HYDROPHONE_ID+'.'+filename
        try:
            shutil.copyfile(os.path.join(dir, filename), os.path.join(path, random_indexed_filename))
        except Exception as e:
            print(e)

## 3. Make a global list of files

For full algorithm runs

In [ ]:
FILE_LIST = r''

In [ ]:
fileinfo_list = []

for input_dir in TIME_PERIODS.values():

    dir_list = [f for f in os.listdir(input_dir) if (f.endswith('.WAV') and f.startswith('20'))]

    for filename in tqdm(dir_list, total = len(dir_list)):

        file_size = os.path.getsize(os.path.join(input_dir, filename))
        in_bounds, _ = isFilenameInPeriods(filename, TIME_PERIODS)

        if in_bounds:
            metadata = parse_metadata(os.path.join(input_dir, filename))
            length_s = metadata.duration_s
            fileinfo_list.append([input_dir, filename, length_s])

In [ ]:
with open(FILE_LIST, 'w') as output:
    json.dump(fileinfo_list, output)